In [1]:
# 1. Setup and Imports
%pip install openai python-dotenv scikit-learn scipy matplotlib scikit-fuzzy --quiet plotly

import os
import numpy as np
import pandas as pd
import openai
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import skfuzzy as fuzz  # for fuzzy c-means

# Load environment variables
load_dotenv()
client = OpenAI()  # Updated API client initialization




[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2. Read the animal names from a text file
with open('animal_names.txt', 'r', encoding='utf-8') as f:
    animals = [line.strip() for line in f if line.strip()]

# Remove duplicates while preserving order
unique_animals = list(dict.fromkeys(animals))

print(f"Number of animal names: {len(unique_animals)}")
unique_animals[:10]  # Show the first 10 for reference


Number of animal names: 574


['Dog',
 'Cat',
 'Bird',
 'Penguin',
 'Eagle',
 'Cockatoo',
 'Camel',
 'Giraffe',
 'Zebra',
 'Lion']

In [3]:
# 3. Define a function to obtain OpenAI embeddings using the latest API
def get_embedding(text, model="text-embedding-3-small"):
    """Fetch embedding for a single piece of text from OpenAI's latest API."""
    text = text.replace("\n", " ")  # Ensure clean input
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding  # Extract embedding vector


In [4]:
# 4. Generate embeddings for each animal name
embeddings = np.array([get_embedding(animal) for animal in unique_animals])

print("Embeddings shape:", embeddings.shape)


Embeddings shape: (574, 1536)


In [5]:
# 5. Compute the distance matrix using cosine distance
distance_matrix = pairwise_distances(embeddings, metric='cosine')
print("Distance matrix shape:", distance_matrix.shape)


Distance matrix shape: (574, 574)


In [21]:
# 6. Perform hierarchical clustering using Ward’s method
Z = linkage(distance_matrix, method='ward')

# Define the number of clusters
NUM_CLUSTERS = 10
cluster_assignments = fcluster(Z, t=NUM_CLUSTERS, criterion='maxclust')

# Store results in a DataFrame
df_hard_clusters = pd.DataFrame({'Animal': unique_animals, 'Cluster': cluster_assignments})
df_hard_clusters.sort_values('Cluster', inplace=True)
df_hard_clusters.reset_index(drop=True, inplace=True)

df_hard_clusters


/var/folders/tj/fdwpmk2x1dj3y17zvq1j9v540000gn/T/ipykernel_57786/1169163012.py:2: ClusterWarning: scipy.cluster: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  Z = linkage(distance_matrix, method='ward')


,Animal,Cluster
0,Orca,1
1,Sharks,1
2,sea lion,1
3,Tiger shark,1
4,Krill,1
...,...,...
569,Rhinoceros,10
570,tortoises,10
571,Iguana,10
572,arowana,10


In [28]:
# Create the interactive dendrogram as before
import plotly.figure_factory as ff

fig = ff.create_dendrogram(
    embeddings,
    orientation='left',
    labels=unique_animals,
    distfun=lambda x: pairwise_distances(x, metric='cosine'),
    linkagefun=lambda x: linkage(x, method='ward')
)
fig.update_layout(width=1200, height=800)

# Save the interactive plot as an HTML file
fig.write_html("interactive_dendrogram.html")


/var/folders/tj/fdwpmk2x1dj3y17zvq1j9v540000gn/T/ipykernel_57786/1406975788.py:9: ClusterWarning:

scipy.cluster: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



In [10]:
# # 8. Fuzzy C-Means Clustering
# # Transpose the embeddings to (features, samples) format
# data_for_fuzzy = embeddings.T

# # Run fuzzy c-means
# cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
#     data_for_fuzzy,
#     c=40,          # Number of clusters
#     m=2.0,        # Fuzziness parameter
#     error=0.005,  # Stopping criterion
#     maxiter=1000,
#     init=None
# )

# # Assign each animal to its most probable cluster
# fuzzy_labels = np.argmax(u, axis=0)

# df_fuzzy_clusters = pd.DataFrame({'Animal': unique_animals, 'Cluster': fuzzy_labels})
# df_fuzzy_clusters.sort_values('Cluster', inplace=True)
# df_fuzzy_clusters.reset_index(drop=True, inplace=True)

# df_fuzzy_clusters.head(20)


,Animal,Cluster
0,chow chow,0
1,polar,0
2,porcupine,0
3,Chipmunk,1
4,tortoise,1
5,lemur,1
6,Chameleon,1
7,armadillo,1
8,sloth,1
9,Bees,2


In [11]:
# 9. Inspect membership degrees for soft clustering
membership_df = pd.DataFrame(u.T, columns=[f"Cluster_{i}" for i in range(u.shape[0])])
membership_df.insert(0, 'Animal', unique_animals)
membership_df.head(20)


,Animal,Cluster_0,Cluster_1,Cluster_2,Cluster_3,Cluster_4,Cluster_5,Cluster_6,Cluster_7,Cluster_8,...,Cluster_30,Cluster_31,Cluster_32,Cluster_33,Cluster_34,Cluster_35,Cluster_36,Cluster_37,Cluster_38,Cluster_39
0,Dog,0.025001,0.025001,0.024999,0.025000,0.024998,0.025001,0.025001,0.025002,0.025000,...,0.024999,0.025000,0.024999,0.024999,0.025000,0.025001,0.025000,0.024999,0.025002,0.025000
1,Cat,0.025001,0.025004,0.025002,0.025001,0.025000,0.025000,0.025000,0.025005,0.024999,...,0.024999,0.025000,0.024996,0.024998,0.024999,0.025004,0.024996,0.024998,0.025003,0.024998
2,Bird,0.025000,0.025002,0.025002,0.025002,0.025003,0.025001,0.024999,0.025003,0.025000,...,0.025000,0.024999,0.024996,0.025000,0.024997,0.025005,0.024999,0.024999,0.025001,0.024997
3,Penguin,0.025002,0.025001,0.024998,0.024999,0.024995,0.025001,0.025000,0.024999,0.024999,...,0.024999,0.025002,0.025003,0.025003,0.025001,0.024999,0.025003,0.024997,0.025002,0.025000
4,Eagle,0.025001,0.025002,0.025001,0.024999,0.024997,0.025001,0.025002,0.025002,0.025001,...,0.024998,0.024999,0.025000,0.024999,0.024999,0.025003,0.025002,0.024999,0.025001,0.025000
5,Cockatoo,0.024998,0.025003,0.025001,0.025001,0.025000,0.025003,0.024996,0.025002,0.024999,...,0.025002,0.025003,0.024998,0.025001,0.025000,0.025002,0.024999,0.024998,0.025000,0.025000
6,Camel,0.025001,0.025004,0.025000,0.024999,0.024994,0.025001,0.025001,0.025002,0.025000,...,0.024999,0.025002,0.025000,0.024999,0.025002,0.025001,0.025002,0.024998,0.025002,0.025001
7,Giraffe,0.024998,0.025004,0.025003,0.024999,0.024990,0.025002,0.024998,0.025001,0.025001,...,0.024999,0.025005,0.025003,0.025002,0.025004,0.025000,0.025003,0.024997,0.025002,0.025001
8,Zebra,0.025001,0.025003,0.025001,0.024999,0.024993,0.025001,0.025001,0.025002,0.025000,...,0.024998,0.025003,0.025001,0.025000,0.025002,0.025000,0.025003,0.024998,0.025002,0.025000
9,Lion,0.025001,0.025005,0.025001,0.024998,0.024988,0.025001,0.025003,0.025003,0.025001,...,0.024995,0.025004,0.025004,0.025000,0.025003,0.025000,0.025003,0.024996,0.025003,0.025001


In [2]:
import re

# --- START OF COPIED DATA (Simulated file content) ---
animal_data = """
Dog
Cat
Bird
Penguin
Eagle
Cockatoo
Camel
Giraffe
Zebra
Lion
Puma
Panther
Jaguar
Cheetah
Tiger
Wolf
Fox
Polar bear
Brown bear
Panda bear
Koala bear
Kangeroo
wombat
quokka
Ants
Bees
Elephant
Anteater
Beaver
Platypus
Ostrich
Swallow
Sparrow
Crow
Dogs
Cats
Monkeys
Walruses
Whales
Sharks
Fishes
Alligators
Zebras
Lions
Giraffes
Wolfs
Rabbit
Fish
Goat
Donkey
Rhinoceros
Horse
Hen
Iguana
Lizard
Leopard
Sea Urchin
Dolphin
Whale
Manatee
orangutan
seal
turtle
tortoise
kangaroo
snake
chimpanzee
hamster
gerbil
rabbits
hamsters
crocodile
hippopotamus
tigers
snakes
birds
leopards
bears
pandas
dinosaurs
chihuahua
pitbull
golden retrievers
mouse
cow
cockroach
monkey
chicken
duck
butterfly
pig
deer
spider
mosquito
hyena
alligator
Ox
Cockroaches
Worms
Centipedes
millipedes
Beetles
Butterflies
Moths
Kangaroos
hawks
robins
koala
polar bears
peacock
flamingo
cod
salmon
grouper
anglerfish
shark
killer whale
great white shark
Bear
Frog
Clownfish
Dory
Chimpanzees
Gorilla
Starfish
Rockfish
pufferfish
panda
sheep
stingray
catepillar
millipede
cricket
grasshopper
moth
Guinea pig
Pigs
Cows
Ant
Eagles
Otter
Parrot
Hawk
Rat
aardvark
boar
civet
falcon
greyhound
yak
seals
bat
squirrel
snail
toad
worm
housefly
emu
racoon
honey badger
dinosaur
grizzly bear
insects
ladybug
alpaca
parrots
Bee
Insect
Beetle
budgie
Reptile
Kingfisher
Owl
monitor lizard
goldfish
otters
wild boar
hornets
Fireflies
Snails
Scorpion
Camels
Humans
Tuna
Salamander
Chameleon
porcupine
kangeroos
sea lion
penguins
rats
horses
Hippo
Lizards
rooster
wolves
flying squirrels
racoons
giant turtles
crocodiles
terrapin
bats
octopus
slugs
clams
finch
koi
prawn
lobster
guppy fish
mole
black bear
donkeys
wasps
termites
beavers
pigeon
goats
Swan
Crab
Akbash
Capybara
Baboon
Caterpillar
Ladybird
Sun bear
Moon bear
Pirana
Catfish
Pigeons
Goose
Dolphins
gecko
pika
chinchilla
wallaby
axolotl
narwhal
dingo
Turkey
elephants
beagles
anteaters
dugongs
guinea pigs
Ploar bear
Blue whale
Giant river otters
Small otter
Woodpecker
Tasmanian devil
Cassowary
Snow leopard
Badger
Swordfish
Narwhals
Belugas
Humpback whales
Sperm whale
Giant squid
Blue-ringed octopus
Box jellyfish
Sea slug
Flatworm
armadillo
sea cucumbers
crabs
prawns
lobsters
mussels
mice
geese
moose
hedgehog
moles
Hippotamus
Seagull
Lamb
Dophin
crows
blue jays
macaws
sunbear
Chipmunk
Hammerhead shark
Dragon fish
red panda
squid
jellyfish
terrapins
spiders
mosquitoes
fruit flies
tortoises
dragonfly
ducks
crayfish
Scallops
Hippos
Rhinoceres
Gorillas
Baboons
dragon
Komodo dragon
hare
seahorse
llama
deers
white tiger
meerkat
pony
hummingbird
liger
squirrels
turtles
Orca
Tiger shark
Raven
Shrimp
Krill
Bluewhale
Beluga whale
Walrus
Apes
kiwi
reindeer
gineau pigs
sea lions
orangutans
antelope
rhino
mule
chickens
Ape
Vulture
Coyote
Centipede
Caterpilla
Wasp
rhinocerous
human
Bull
Hornbill
Chinchillas
Sloths
Flamingos
Jabiru
roach
buffalo
Stick insect
Unicorn
Cobra
chicks
cheetahs
porpupines
quokkas
guppy
trinoceros
T-rex
abalone
chick
calf
komodo dragons
ligers
hippotomas
sparrows
angler fish
mynah birds
Clown fish
boars
panthers
mealworm
earthworm
python
anaconda
Hippoptamus
Wildcat
Sheeps
fly
Wildboar
Goost
Koi fish
Stonefish
Frogfish
Prairie dog
cougar
hyenas
Pangolin
Hump back whale
Whale shark
Seagulls
croc
anteter
crcodile
trout
Quail
bunny
Porpoise
Kitten
scorpions
foxes
tapirs
Slug
crane
lemur
mongoose
Clam
rabits
guniea pigs
Oysters
Bulls
Meerkats
Honey badgers
Codfish
Sea Cucumber
Seasnail
eels
swans
hummingbirds
myna
mockingbird
cuckoobirds
crcocodiles
Frogs
rhinos
raccoon
wild dog
sloth
dugong
pike
Cobras
tapir
gorrilla
chipannzee
rhinosaurus
skunk
aligator
tortise
drop bear
hares
lady bugs
Hippopotamu
Newt
Chinpanzee
Budgerigar
gazelle
polar
hyenna
giraff
bull whale
naked mole rat
vole
dodo bird
chincilla
sunfish
pelican
Kaola
Sea anemone
White pomfret
Sardines
echidna
jaguars
Sealion
Rrocodile
Flies
vulcan
maggot
kite
gibbon
Stringray
Komono dragon
Lama
Mountain goat
Minks
doves
porcupines
mara
Golden retreiver
Vultures
Tarantulas
Kiwis
Koalas
Stingrays
Toads
seahorses
Crocs
Wombats
Coyotes
Armadillos
Hedgehogs
cuttlefish
flying fox
egret
albatross
orang utan
snowleopard
Weasel
Beluga
Stork
anteears
jackals
mynas
arctic fox
mole rat
Playtpus
Carp
golden retriever
corgi
japanese spitz
persian cat
ginger tabby cat
cocker spaniel
hound dog
dobermann
chow chow
red pandas
raccoons
brown bears
monitor lizards
Dove
eel
pengins
wilderbeast
bettle
Chincillas
Hens
Plankton
robin
hornet
zebra fish
arowana
owls
mosquitos
houseflies
cheetas
snow leopards
gorrillas
piranha
Termite
Milipede
Mynah
Golden Retriver
Sardine
lionfish
ostritch
viper
flammingo
tigress
baracuda
sea anemane
mantis
barnacle
hermit crab
horseshoe crab
flounder
parrot fish
needle fish
crocodile fish
sea star
Ponies
meercat
russion blue cat
parakeet
bobcat
Catepillars
Snapper
Cannary
aarddvark
TARANTULA
SABERTOOTH TIGER
Mammals
Sabertooth
ANGELFISH
BOXFISH
"""
# --- END OF COPIED DATA ---

# Manual mapping for known misspellings, irregular plurals, and desired standardizations
# Format: 'variation_lowercase': 'Canonical Name'
manual_corrections = {
    'kangeroo': 'Kangaroo',
    'ants': 'Ant',
    'bees': 'Bee',
    'dogs': 'Dog',
    'cats': 'Cat',
    'monkeys': 'Monkey',
    'walruses': 'Walrus',
    'whales': 'Whale',
    'sharks': 'Shark',
    'fishes': 'Fish', # map to singular 'Fish'
    'alligators': 'Alligator',
    'zebras': 'Zebra',
    'lions': 'Lion',
    'giraffes': 'Giraffe',
    'wolfs': 'Wolf', # specific misspelling of plural
    'wolves': 'Wolf',
    'orangutan': 'Orangutan',
    'rabbits': 'Rabbit',
    'hamsters': 'Hamster',
    'tigers': 'Tiger',
    'snakes': 'Snake',
    'birds': 'Bird',
    'leopards': 'Leopard',
    'bears': 'Bear',
    'pandas': 'Panda',
    'dinosaurs': 'Dinosaur',
    'golden retrievers': 'Golden Retriever', # Standardize capitalization
    'cockroaches': 'Cockroach',
    'worms': 'Worm',
    'centipedes': 'Centipede',
    'millipedes': 'Millipede',
    'beetles': 'Beetle',
    'butterflies': 'Butterfly',
    'moths': 'Moth',
    'kangaroos': 'Kangaroo',
    'hawks': 'Hawk',
    'robins': 'Robin',
    'koala': 'Koala',
    'polar bears': 'Polar Bear',
    'chimpanzees': 'Chimpanzee',
    'pigs': 'Pig',
    'cows': 'Cow',
    'eagles': 'Eagle',
    'seals': 'Seal',
    'parrots': 'Parrot',
    'rats': 'Rat',
    'racoon': 'Raccoon', # spelling
    'racoons': 'Raccoon', # spelling + plural
    'insects': 'Insect',
    'budgie': 'Budgerigar', # map to more formal name
    'otters': 'Otter',
    'wild boar': 'Wild Boar',
    'hornets': 'Hornet',
    'fireflies': 'Firefly',
    'snails': 'Snail',
    'camels': 'Camel',
    'humans': 'Human',
    'kangeroos': 'Kangaroo', # spelling
    'sea lion': 'Sea Lion',
    'sea lions': 'Sea Lion',
    'penguins': 'Penguin',
    'horses': 'Horse',
    'hippo': 'Hippopotamus', # map abbreviation
    'hippos': 'Hippopotamus',
    'lizards': 'Lizard',
    'flying squirrels': 'Flying Squirrel',
    'giant turtles': 'Giant Turtle',
    'crocodiles': 'Crocodile',
    'bats': 'Bat',
    'slugs': 'Slug',
    'clams': 'Clam',
    'koi': 'Koi Fish', # be more specific
    'prawn': 'Prawn',
    'prawns': 'Prawn',
    'lobster': 'Lobster',
    'lobsters': 'Lobster',
    'guppy fish': 'Guppy', # simplify
    'donkeys': 'Donkey',
    'wasps': 'Wasp',
    'termites': 'Termite',
    'beavers': 'Beaver',
    'pigeons': 'Pigeon',
    'goats': 'Goat',
    'pirana': 'Piranha', # spelling
    'dolphins': 'Dolphin',
    'elephants': 'Elephant',
    'beagles': 'Beagle',
    'anteaters': 'Anteater',
    'dugongs': 'Dugong',
    'guinea pigs': 'Guinea Pig',
    'ploar bear': 'Polar Bear', # spelling
    'blue whale': 'Blue Whale',
    'giant river otters': 'Giant River Otter',
    'small otter': 'Otter', # map to general otter unless specific needed
    'tasmanian devil': 'Tasmanian Devil',
    'snow leopard': 'Snow Leopard',
    'snow leopards': 'Snow Leopard',
    'narwhals': 'Narwhal',
    'belugas': 'Beluga Whale', # be more specific
    'beluga': 'Beluga Whale',
    'humpback whales': 'Humpback Whale',
    'sperm whale': 'Sperm Whale',
    'giant squid': 'Giant Squid',
    'blue-ringed octopus': 'Blue-Ringed Octopus',
    'box jellyfish': 'Box Jellyfish',
    'sea slug': 'Sea Slug',
    'flatworm': 'Flatworm',
    'sea cucumbers': 'Sea Cucumber',
    'crabs': 'Crab',
    'mussels': 'Mussel',
    'mice': 'Mouse',
    'geese': 'Goose',
    'moles': 'Mole',
    'hippotamus': 'Hippopotamus', # spelling
    'dophin': 'Dolphin', # spelling
    'crows': 'Crow',
    'blue jays': 'Blue Jay',
    'macaws': 'Macaw',
    'sunbear': 'Sun Bear', # spacing
    'hammerhead shark': 'Hammerhead Shark',
    'dragon fish': 'Dragonfish', # combine? Or keep separate? Let's keep separate: 'Dragon Fish'
    'red panda': 'Red Panda',
    'red pandas': 'Red Panda',
    'terrapins': 'Terrapin',
    'spiders': 'Spider',
    'mosquitoes': 'Mosquito',
    'fruit flies': 'Fruit Fly',
    'tortoises': 'Tortoise',
    'ducks': 'Duck',
    'scallops': 'Scallop',
    'rhinoceres': 'Rhinoceros', # spelling
    'gorillas': 'Gorilla',
    'baboons': 'Baboon',
    'komodo dragon': 'Komodo Dragon',
    'komodo dragons': 'Komodo Dragon',
    'deers': 'Deer', # irregular plural
    'white tiger': 'White Tiger',
    'liger': 'Liger',
    'ligers': 'Liger',
    'squirrels': 'Squirrel',
    'turtles': 'Turtle',
    'bluewhale': 'Blue Whale', # spacing
    'beluga whale': 'Beluga Whale',
    'apes': 'Ape',
    'gineau pigs': 'Guinea Pig', # spelling
    'orangutans': 'Orangutan',
    'chickens': 'Chicken',
    'caterpilla': 'Caterpillar', # spelling
    'rhinocerous': 'Rhinoceros', # spelling
    'human': 'Human',
    'chinchillas': 'Chinchilla',
    'sloths': 'Sloth',
    'flamingos': 'Flamingo',
    'roach': 'Cockroach', # map abbreviation
    'stick insect': 'Stick Insect',
    'cobra': 'Cobra',
    'cobras': 'Cobra',
    'chicks': 'Chick', # map plural
    'cheetahs': 'Cheetah',
    'porpupines': 'Porcupine', # spelling
    'porcupines': 'Porcupine',
    'quokkas': 'Quokka',
    'guppy': 'Guppy',
    'trinoceros': 'Rhinoceros', # spelling
    't-rex': 'T-Rex', # Capitalization
    'hippotomas': 'Hippopotamus', # spelling
    'sparrows': 'Sparrow',
    'angler fish': 'Anglerfish', # Combine
    'mynah birds': 'Mynah', # simplify
    'clown fish': 'Clownfish', # Combine
    'boars': 'Boar',
    'panthers': 'Panther',
    'hippoptamus': 'Hippopotamus', # spelling
    'sheeps': 'Sheep', # irregular plural
    'wildboar': 'Wild Boar', # spacing
    'goost': 'Goose', # spelling
    'koi fish': 'Koi Fish',
    'stonefish': 'Stonefish',
    'frogfish': 'Frogfish',
    'prairie dog': 'Prairie Dog',
    'hump back whale': 'Humpback Whale', # spacing
    'whale shark': 'Whale Shark',
    'seagulls': 'Seagull',
    'croc': 'Crocodile', # abbreviation
    'anteter': 'Anteater', # spelling
    'crcodile': 'Crocodile', # spelling
    'crcocodiles': 'Crocodile', # spelling + plural
    'rabits': 'Rabbit', # spelling
    'guniea pigs': 'Guinea Pig', # spelling
    'oysters': 'Oyster',
    'bulls': 'Bull',
    'meerkats': 'Meerkat',
    'honey badgers': 'Honey Badger',
    'codfish': 'Cod', # simplify
    'sea cucumber': 'Sea Cucumber',
    'seasnail': 'Sea Snail', # spacing
    'eels': 'Eel',
    'swans': 'Swan',
    'hummingbirds': 'Hummingbird',
    'myna': 'Mynah', # standard spelling
    'mynas': 'Mynah',
    'cuckoobirds': 'Cuckoo', # simplify
    'frogs': 'Frog',
    'rhinos': 'Rhinoceros', # map abbreviation + plural
    'rhino': 'Rhinoceros', # map abbreviation
    'wild dog': 'Wild Dog', # or African Wild Dog? Keep simple for now.
    'gorrilla': 'Gorilla', # spelling
    'chipannzee': 'Chimpanzee', # spelling
    'chinpanzee': 'Chimpanzee', # spelling
    'rhinosaurus': 'Rhinoceros', # spelling
    'aligator': 'Alligator', # spelling
    'tortise': 'Tortoise', # spelling
    'drop bear': 'Drop Bear', # Assuming this is intentional :)
    'hares': 'Hare',
    'lady bugs': 'Ladybug', # combine
    'hippopotamu': 'Hippopotamus', # spelling
    'polar': 'Polar Bear', # Assuming context implies Polar Bear
    'hyenna': 'Hyena', # spelling
    'giraff': 'Giraffe', # spelling
    'bull whale': 'Whale', # Simplify - could be ambiguous, map to general Whale
    'naked mole rat': 'Naked Mole Rat',
    'chincilla': 'Chinchilla', # spelling
    'kaola': 'Koala', # spelling
    'sea anemone': 'Sea Anemone',
    'white pomfret': 'White Pomfret',
    'sardines': 'Sardine',
    'jaguars': 'Jaguar',
    'sealion': 'Sea Lion', # spacing
    'rrocodile': 'Crocodile', # spelling
    'flies': 'Fly',
    'vulcan': 'Vulcan', # Fictional? Keep as is unless context differs.
    'stringray': 'Stingray', # spelling
    'stingrays': 'Stingray',
    'komono dragon': 'Komodo Dragon', # spelling
    'lama': 'Llama', # spelling
    'mountain goat': 'Mountain Goat',
    'minks': 'Mink',
    'doves': 'Dove',
    'golden retreiver': 'Golden Retriever', # spelling
    'vultures': 'Vulture',
    'tarantulas': 'Tarantula',
    'kiwis': 'Kiwi',
    'koalas': 'Koala',
    'toads': 'Toad',
    'seahorses': 'Seahorse',
    'crocs': 'Crocodile', # abbreviation
    'wombats': 'Wombat',
    'coyotes': 'Coyote',
    'armadillos': 'Armadillo',
    'hedgehogs': 'Hedgehog',
    'orang utan': 'Orangutan', # spacing
    'snowleopard': 'Snow Leopard', # spacing
    'anteears': 'Anteater', # spelling
    'jackals': 'Jackal',
    'arctic fox': 'Arctic Fox',
    'mole rat': 'Mole Rat',
    'playtpus': 'Platypus', # spelling
    'golden retriever': 'Golden Retriever',
    'japanese spitz': 'Japanese Spitz',
    'persian cat': 'Persian Cat',
    'ginger tabby cat': 'Ginger Tabby Cat',
    'cocker spaniel': 'Cocker Spaniel',
    'hound dog': 'Hound', # simplify? Or keep 'Hound Dog'? Keep for now.
    'chow chow': 'Chow Chow',
    'monitor lizards': 'Monitor Lizard',
    'pengins': 'Penguin', # spelling
    'wilderbeast': 'Wildebeest', # spelling
    'bettle': 'Beetle', # spelling
    'chincillas': 'Chinchilla', # spelling + plural
    'hens': 'Hen',
    'zebra fish': 'Zebrafish', # combine
    'mosquitos': 'Mosquito', # spelling
    'houseflies': 'Housefly',
    'cheetas': 'Cheetah', # spelling
    'gorrillas': 'Gorilla', # spelling + plural
    'piranha': 'Piranha',
    'milipede': 'Millipede', # spelling
    'mynah': 'Mynah',
    'golden retriver': 'Golden Retriever', # spelling
    'sardine': 'Sardine',
    'ostritch': 'Ostrich', # spelling
    'flammingo': 'Flamingo', # spelling
    'baracuda': 'Barracuda', # spelling
    'sea anemane': 'Sea Anemone', # spelling
    'hermit crab': 'Hermit Crab',
    'horseshoe crab': 'Horseshoe Crab',
    'parrot fish': 'Parrot Fish',
    'needle fish': 'Needlefish', # combine
    'crocodile fish': 'Crocodilefish', # combine
    'sea star': 'Starfish', # Common name preference
    'ponies': 'Pony',
    'meercat': 'Meerkat', # spelling
    'russion blue cat': 'Russian Blue Cat', # spelling + capitalization
    'catepillars': 'Caterpillar', # spelling + plural
    'cannary': 'Canary', # spelling
    'aarddvark': 'Aardvark', # spelling
    'tarantula': 'Tarantula',
    'sabertooth tiger': 'Saber-Toothed Tiger', # Hyphenation & Capitalization
    'sabertooth': 'Saber-Toothed Tiger', # Map incomplete name
    'mammals': 'Mammal', # Generic category, map to singular
    'angelfish': 'Angelfish',
    'boxfish': 'Boxfish',
    'reptile': 'Reptile', # Generic category
    'great white shark': 'Great White Shark',
    'killer whale': 'Orca', # Common name preference
    'clownfish': 'Clownfish',
    'pufferfish': 'Pufferfish',
    'guinea pig': 'Guinea Pig',
    'honey badger': 'Honey Badger',
    'grizzly bear': 'Grizzly Bear',
    'ladybird': 'Ladybug', # Synonym preference
    'sun bear': 'Sun Bear',
    'moon bear': 'Moon Bear',
    'catfish': 'Catfish',
    'blue jay': 'Blue Jay',
    'tiger shark': 'Tiger Shark',
}

# Function to generate a canonical name (simple version)
# Handles capitalization and checks manual corrections first
def get_canonical_name(name):
    name = name.strip()
    if not name:
        return None
    
    lower_name = name.lower()
    
    # 1. Check manual corrections first (most specific)
    if lower_name in manual_corrections:
        return manual_corrections[lower_name]
        
    # 2. Basic pluralization check (ends in 's', not 'ss', etc.)
    # This is very basic and might misfire, manual corrections are better
    # Only apply if the singular form isn't explicitly in manual corrections
    # (e.g., don't turn 'walrus' into 'walru')
    # Avoid changing words like 'bus', 'glass', 'species' etc.
    singular = lower_name
    if lower_name.endswith('s') and not lower_name.endswith(('ss', 'us', 'is', 'es')):
         singular_test = lower_name[:-1]
         # If the potential singular form exists in manual corrections, use ITS canonical form
         if singular_test in manual_corrections:
              return manual_corrections[singular_test]
         # Otherwise, tentatively assume this is the singular form (needs Title Case later)
         singular = singular_test
    elif lower_name.endswith('es') and len(lower_name) > 3 and lower_name[-3] not in 'aeioush': # handles foxes, but not bees or horses
        singular_test = lower_name[:-2]
        if singular_test in manual_corrections:
            return manual_corrections[singular_test]
        singular = singular_test
    elif lower_name.endswith('ies') and len(lower_name) > 3:
         singular_test = lower_name[:-3] + 'y'
         if singular_test in manual_corrections:
             return manual_corrections[singular_test]
         singular = singular_test
    else:
        singular = lower_name # Use the original lowercase if no simple plural rule applied

    # 3. Apply Title Case / Capitalization
    # If the singular form (after potential basic depluralization) has a manual entry, use it
    if singular in manual_corrections:
        return manual_corrections[singular]

    # Otherwise, apply standard capitalization rules
    words = singular.split(' ')
    # Simple capitalize for single words, title case for multiple
    if len(words) == 1:
         # Capitalize first letter, unless it's something like T-Rex
         if '-' in words[0] and len(words[0]) > 1: # Handle T-Rex like cases
             return '-'.join(part.capitalize() for part in words[0].split('-'))
         return words[0].capitalize()
    else:
        # Title case for multi-word names, handling hyphens within words
        cased_words = []
        for word in words:
            if '-' in word and len(word) > 1:
                 cased_words.append('-'.join(part.capitalize() for part in word.split('-')))
            else:
                cased_words.append(word.capitalize())
        return ' '.join(cased_words)


# --- Main Processing Logic ---

# Get unique, non-empty names from the input
raw_names = set(name.strip() for name in animal_data.strip().split('\n') if name.strip())

# Dictionary to hold the final mapping
animal_dict = {}

# Dictionary to group variations under their canonical name temporarily
canonical_groups = {}

# First pass: determine canonical name for each unique raw name
for name in raw_names:
    canonical = get_canonical_name(name)
    if canonical:
        if canonical not in canonical_groups:
            canonical_groups[canonical] = set()
        # Store the original variation that led to this canonical name
        canonical_groups[canonical].add(name)

# Second pass: build the final dictionary mapping all variations to the canonical name
for canonical, variations in canonical_groups.items():
    # Add the canonical name itself mapping to itself
    animal_dict[canonical] = canonical
    # Add all variations mapping to the canonical name
    for variation in variations:
        # Ensure we don't overwrite the canonical mapping if a variation happens to be the same string
        if variation != canonical:
             animal_dict[variation] = canonical
        # Add lowercase version mapping to canonical, if different
        lower_variation = variation.lower()
        if lower_variation != canonical.lower() and lower_variation not in animal_dict:
            animal_dict[lower_variation] = canonical


# --- Output ---

# Print the resulting dictionary (optional, can be large)
import pprint
pprint.pprint(animal_dict)

# Example usage:
print(f"Mapping for 'Dogs': {animal_dict.get('Dogs')}")
print(f"Mapping for 'dogs': {animal_dict.get('dogs')}")
print(f"Mapping for 'Dog': {animal_dict.get('Dog')}")
print(f"Mapping for 'Kangeroo': {animal_dict.get('Kangeroo')}")
print(f"Mapping for 'kangeroo': {animal_dict.get('kangeroo')}")
print(f"Mapping for 'Kangaroo': {animal_dict.get('Kangaroo')}")
print(f"Mapping for 'Polar bear': {animal_dict.get('Polar bear')}")
print(f"Mapping for 'polar bears': {animal_dict.get('polar bears')}")
print(f"Mapping for 'Ploar bear': {animal_dict.get('Ploar bear')}")
print(f"Mapping for 'Sea Lion': {animal_dict.get('Sea Lion')}")
print(f"Mapping for 'sea lions': {animal_dict.get('sea lions')}")
print(f"Mapping for 'Sealion': {animal_dict.get('Sealion')}")
print(f"Mapping for 'rhinocerous': {animal_dict.get('rhinocerous')}")
print(f"Mapping for 'Rhino': {animal_dict.get('Rhino')}")
print(f"Mapping for 'rhinos': {animal_dict.get('rhinos')}")
print(f"Mapping for 'crcodile': {animal_dict.get('crcodile')}")
print(f"Mapping for 'Croc': {animal_dict.get('Croc')}")
print(f"Mapping for 'mice': {animal_dict.get('mice')}")
print(f"Mapping for 'geese': {animal_dict.get('geese')}")
print(f"Mapping for 'golden retreiver': {animal_dict.get('golden retreiver')}")
print(f"Mapping for 'Golden Retriever': {animal_dict.get('Golden Retriever')}")
print(f"Mapping for 'T-rex': {animal_dict.get('T-rex')}")
print(f"Mapping for 't-rex': {animal_dict.get('t-rex')}")

{'ANGELFISH': 'Angelfish',
 'Aardvark': 'Aardvark',
 'Abalone': 'Abalone',
 'Akbash': 'Akbash',
 'Albatross': 'Albatross',
 'Alligator': 'Alligator',
 'Alligators': 'Alligator',
 'Alpaca': 'Alpaca',
 'Anaconda': 'Anaconda',
 'Angelfish': 'Angelfish',
 'Anglerfish': 'Anglerfish',
 'Ant': 'Ant',
 'Anteater': 'Anteater',
 'Antelope': 'Antelope',
 'Ants': 'Ant',
 'Ape': 'Ape',
 'Apes': 'Ape',
 'Arctic Fox': 'Arctic Fox',
 'Armadillo': 'Armadillo',
 'Armadillos': 'Armadillo',
 'Arowana': 'Arowana',
 'Axolotl': 'Axolotl',
 'BOXFISH': 'Boxfish',
 'Baboon': 'Baboon',
 'Baboons': 'Baboon',
 'Badger': 'Badger',
 'Barnacle': 'Barnacle',
 'Barracuda': 'Barracuda',
 'Bat': 'Bat',
 'Beagle': 'Beagle',
 'Bear': 'Bear',
 'Beaver': 'Beaver',
 'Bee': 'Bee',
 'Bees': 'Bee',
 'Beetle': 'Beetle',
 'Beetles': 'Beetle',
 'Beluga': 'Beluga Whale',
 'Beluga Whale': 'Beluga Whale',
 'Beluga whale': 'Beluga Whale',
 'Belugas': 'Beluga Whale',
 'Bird': 'Bird',
 'Black Bear': 'Black Bear',
 'Blue Jay': 'Blue Jay',